# Climate data science

This notebook runs all analyses for Assignment 2.

**Contents:**
- Question 1: 

**Global Seed:** All scripts use consistent seeds for reproducibility.

In [24]:
# Setup: Add src to path
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# for automatic reloading of modules during development
%load_ext autoreload
%autoreload 2


# Add src directory to Python path
notebook_dir = Path.cwd()
src_path = notebook_dir / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Notebook directory: {notebook_dir}")
print(f"Source directory: {src_path}")
print(f"Source directory exists: {src_path.exists()}")
print(f"Succesfully configured the paths.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Notebook directory: /Users/PacoSmit/Desktop/VU/Master/Climate Data Science/CDS_assignment2
Source directory: /Users/PacoSmit/Desktop/VU/Master/Climate Data Science/CDS_assignment2/src
Source directory exists: True
Succesfully configured the paths.


## Question 1: Static SAR models

In [25]:
# Import the functions and run Question 1
from src.q1_a import *
from src.q1_b import *
from src.q1_d import *

def main_q1():
    # here you run the code for question 1 
    # by calling the functions you defined in q1_a and q1_b and q1_d
    pass

main_q1()


## Question 2: dynamic spatial models with homogeneous spatial effects.

In [ ]:
# Import and run Question 2
from src.q2_a import (
    load_data,
    link,
    run_filter,
    estimate_gaussian_model,
    qlr_test,
    plot_rho_path,
    plot_rho_comparison,
    print_estimates,
    print_qlr_results,
    K_REGRESSORS,
)

os.makedirs('figures', exist_ok=True)

# Data paths
CDS_PATH     = 'data/cds_data.xlsx'
WEIGHTS_PATH = 'data/cds_spatialweights.xlsx'


def main_q2a():
    """
    Run all analyses for Question 2(a):
      1. Estimate the Gaussian score-driven model under the main specification.
      2. Estimate two alternative specifications and compare.
      3. Perform the QLR test for absence of score-driven dynamics.
      4. Produce figures.
    """

    # 1. Load data
    Y, Xt_all, W, WY_all, _ = load_data(CDS_PATH, WEIGHTS_PATH)
    T, n = Y.shape
    k    = K_REGRESSORS
    print(f"  T = {T},  n = {n},  k = {k}")

    # 2. Main specification  (lecture slides p.55)
    #    B = [0, 0.99],  alpha in [0, 1],  omega in [0, 2],  sigma in [1, 10]
    print("MAIN SPECIFICATION  (slides p.55)")
    print("  B = [0, 0.99],  Theta_alpha = [0, 1]")
    print("  h(f) = 0.95 * tanh(f),  sigma in [1, 10]")

    bounds_main = (
        [(0.0,          2.0),           # omega  in [0, 2]
         (0.0,          0.99),          # beta   in B = [0, 0.99]
         (0.0,          1.0),           # alpha  in Theta_alpha = [0, 1]
         (np.log(1.0),  np.log(10.0))] # sigma  in [1, 10]  (via log_sigma)
        + [(-20.0, 20.0)] * k          # beta_reg unconstrained
    )

    params_main, ll_main, _ = estimate_gaussian_model(
        Y, Xt_all, W, WY_all, bounds_main
    )
    _, f_main, rho_main = run_filter(
        *params_main[:4], params_main[4:], Y, Xt_all, W, WY_all
    )

    print_estimates("Main specification estimates:", params_main, ll_main, rho_main)

    # 3. Alternative specification 1: wider alpha  [0, 2]
    print("ALTERNATIVE SPECIFICATION 1")
    print("  B = [0, 0.99],  Theta_alpha = [0, 2]")

    bounds_alt1 = (
        [(0.0,         2.0),
         (0.0,         0.99),
         (0.0,         2.0),           # wider alpha
         (np.log(1.0), np.log(10.0))]
        + [(-20.0, 20.0)] * k
    )

    params_alt1, ll_alt1, _ = estimate_gaussian_model(
        Y, Xt_all, W, WY_all, bounds_alt1
    )
    _, f_alt1, rho_alt1 = run_filter(
        *params_alt1[:4], params_alt1[4:], Y, Xt_all, W, WY_all
    )

    print_estimates("Alternative 1 estimates:", params_alt1, ll_alt1, rho_alt1)

    # 4. Alternative specification 2: wider beta  [0, 0.999]
    print("ALTERNATIVE SPECIFICATION 2")
    print("  B = [0, 0.999],  Theta_alpha = [0, 1]")

    bounds_alt2 = (
        [(0.0,         2.0),
         (0.0,         0.999),         # wider beta
         (0.0,         1.0),
         (np.log(1.0), np.log(10.0))]
        + [(-20.0, 20.0)] * k
    )

    params_alt2, ll_alt2, _ = estimate_gaussian_model(
        Y, Xt_all, W, WY_all, bounds_alt2
    )
    _, f_alt2, rho_alt2 = run_filter(
        *params_alt2[:4], params_alt2[4:], Y, Xt_all, W, WY_all
    )

    print_estimates("Alternative 2 estimates:", params_alt2, ll_alt2, rho_alt2)

    # 5. QLR test for time variation  (H0: alpha = 0)
    #    Bounds for restricted model: [omega, beta (irrelevant), log_sigma, beta_reg]
    print("QLR TEST FOR TIME VARIATION")

    bounds_restricted = (
        [(0.0,         2.0),
         (0.0,         0.99),          # beta present but irrelevant under H0
         (np.log(1.0), np.log(10.0))]
        + [(-20.0, 20.0)] * k
    )

    qlr_results = qlr_test(
        ll_main, params_main,
        Y, Xt_all, W, WY_all,
        bounds_restricted,
        beta_U=0.99, alpha_L=0,
    )

    print_qlr_results(qlr_results)

    rho_const = qlr_results['rho_const']

    # 6. Figures
    print("\nGenerating figures...")

    # Figure 1: replicate slide p.56
    plot_rho_path(
        rho_main, rho_const, T,
        save_path='figures/q2a_rho_main.png'
    )

    # Figure 2: comparison of parameter space specifications
    labels = [
        f"Main: B=[0,0.99], α∈[0,1]\n"
        f"α={params_main[2]:.4f}, β={params_main[1]:.4f}, σ={np.exp(params_main[3]):.2f}",

        f"Alt 1: α∈[0,2]\n"
        f"α={params_alt1[2]:.4f}, β={params_alt1[1]:.4f}, σ={np.exp(params_alt1[3]):.2f}",

        f"Alt 2: B=[0,0.999]\n"
        f"α={params_alt2[2]:.4f}, β={params_alt2[1]:.4f}, σ={np.exp(params_alt2[3]):.2f}",
    ]

    plot_rho_comparison(
        [rho_main, rho_alt1, rho_alt2], labels, rho_const,
        save_path='figures/q2a_rho_comparison.png'
    )

    # 7. Final summary
    print("SUMMARY — QUESTION 2(a)")
    print(f"{'Specification':<30} {'omega':>8} {'beta':>8} {'alpha':>8} "
          f"{'sigma':>8} {'LL':>14}")
    print("-" * 80)
    for label, p, ll in [
        ("Main (B=[0,.99], α∈[0,1])",  params_main, ll_main),
        ("Alt 1 (α∈[0,2])",             params_alt1, ll_alt1),
        ("Alt 2 (B=[0,.999])",           params_alt2, ll_alt2),
    ]:
        print(f"{label:<30} {p[0]:>8.4f} {p[1]:>8.4f} {p[2]:>8.4f} "
              f"{np.exp(p[3]):>8.4f} {ll:>14.4f}")

    r = qlr_results
    print(f"\nQLR test (main spec):")
    print(f"  QLRT      = {r['QLRT']:.4f}")
    print(f"  kappa_hat = {r['kappa_hat']:.6f}")
    print(f"  QLR_tilde = {r['QLR_tilde']:.4f}")
    print(f"  CV (1%)   = {r['cv_01']}")
    print(f"  {r['conclusion']}")

    print("QUESTION 2(a) COMPLETE")

    
# from src.q2_d import *
# from src.q2_e import *


main_q2a()

ImportError: cannot import name 'function_q2_a' from 'q2_a' (/Users/PacoSmit/Desktop/VU/Master/Climate Data Science/CDS_assignment2/src/q2_a.py)

## Code for Q2(c) (temporary; so we don't have to run the entire code each time)

In [28]:
from src.q2_c import (
    SpatialScoreModel, 
    simulate_spatial_data,
    compute_aic,
    compute_bic
)

def main_q2c():

    output_dir = Path("figures_q2")
    output_dir.mkdir(exist_ok=True)

    # Set random seed for reproducibility
    np.random.seed(42)

    # Simulation Parameters (following Section 3 of the paper)
    T = 500  # Time periods
    n = 9    # Cross-sectional units
    k = 3    # Number of regressors (including intercept)

    # Create spatial weights matrix (row-normalized)
    W_raw = np.random.rand(n, n)
    np.fill_diagonal(W_raw, 0)
    W = W_raw / W_raw.sum(axis=1, keepdims=True)  # Row-normalize

    # True parameters
    omega_true = 0.05
    A_true = 0.05
    B_true = 0.80
    beta_true = np.array([0.0, 1.5, -0.5])  # [intercept, beta1, beta2]
    sigma2_true = 2.0
    nu_true = 5.0

    # Generate time-varying rho process (sine wave pattern)
    t_grid = np.arange(T)
    rho_true = 0.5 + 0.4 * np.cos(2 * np.pi * t_grid / 200)

    print("Question 2c: Time-Varying Spatial Score Model")
    print(f"\nSimulation Setup:")
    print(f"  Time periods (T): {T}")
    print(f"  Cross-sections (n): {n}")
    print(f"  True parameters: ω={omega_true}, A={A_true}, B={B_true}")
    print(f"  Student's t df: {nu_true}")

    # Simulate Data
    print("Simulating data from spatial model...")

    y_data, X_data = simulate_spatial_data(
        T, n, W, rho_true, beta_true, sigma2_true, nu_true
    )

    print(f"  Data shape: y={y_data.shape}, X={X_data.shape}")

    # Estimate Model
    print("Estimating spatial score model...")

    model = SpatialScoreModel(max_rho=0.99)
    result = model.fit(y_data, X_data, W, verbose=True)

    # Display results
    print("Estimation Results:")
    print(f"  Convergence: {'Yes' if result.success else 'No'}")
    print(f"  Log-likelihood: {-result.fun:.2f}")
    print(f"\nEstimated Parameters:")
    print(f"  ω (omega):  {model.params['omega']:.4f}  (true: {omega_true:.4f})")
    print(f"  A:          {model.params['A']:.4f}  (true: {A_true:.4f})")
    print(f"  B:          {model.params['B']:.4f}  (true: {B_true:.4f})")
    print(f"  β:          {model.params['beta']}")
    print(f"  σ²:         {model.params['sigma2']:.4f}  (true: {sigma2_true:.4f})")
    print(f"  ν:          {model.params['nu']:.4f}  (true: {nu_true:.4f})")

    # Unconditional mean
    rho_unconditional = model.get_unconditional_mean()
    print(f"\n  Unconditional mean of ρ: {rho_unconditional:.4f}")

    # Model selection criteria
    n_params = len(result.x)
    aic = compute_aic(-result.fun, n_params)
    bic = compute_bic(-result.fun, n_params, T * n)
    print(f"\n  AIC: {aic:.2f}")
    print(f"  BIC: {bic:.2f}")

    # Visualization
    print("Creating visualizations...")

    # Figure 1: True vs Filtered rho
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(rho_true, 'k-', linewidth=2, label='True ρ(t)', alpha=0.7)
    ax.plot(model.filtered_rho, 'r--', linewidth=2, label='Filtered ρ(t)', alpha=0.7)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Spatial Dependence (ρ)', fontsize=12)
    ax.set_title('Time-Varying Spatial Dependence: True vs Filtered', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / "q2c_rho_comparison.png", dpi=300, bbox_inches='tight')
    print(f"  Saved: {output_dir / 'q2c_rho_comparison.png'}")
    plt.close()

    # Figure 2: Multiple panels showing model components
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Panel 1: Filtered rho
    axes[0, 0].plot(model.filtered_rho, 'b-', linewidth=1.5)
    axes[0, 0].set_xlabel('Time', fontsize=11)
    axes[0, 0].set_ylabel('ρ(t)', fontsize=11)
    axes[0, 0].set_title('Filtered Spatial Dependence', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axhline(y=rho_unconditional, color='r', linestyle='--', 
                        label=f'Unconditional mean: {rho_unconditional:.3f}')
    axes[0, 0].legend(fontsize=9)

    # Panel 2: Distribution of rho
    axes[0, 1].hist(model.filtered_rho, bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(x=rho_unconditional, color='r', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('ρ(t)', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('Distribution of Spatial Dependence', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')

    # Panel 3: Sample of y_data (first 3 units)
    for i in range(min(3, n)):
        axes[1, 0].plot(y_data[:, i], alpha=0.7, label=f'Unit {i+1}')
    axes[1, 0].set_xlabel('Time', fontsize=11)
    axes[1, 0].set_ylabel('y', fontsize=11)
    axes[1, 0].set_title('Sample Time Series (First 3 Units)', fontsize=12, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)

    # Panel 4: Residual analysis
    residuals = np.zeros((T, n))
    for t in range(T):
        rho_t = model.filtered_rho[t]
        y_t = y_data[t]
        X_t = X_data[t]
        residuals[t] = y_t - rho_t * (W @ y_t) - X_t @ model.params['beta']

    axes[1, 1].plot(residuals.mean(axis=1), 'g-', linewidth=1.5)
    axes[1, 1].set_xlabel('Time', fontsize=11)
    axes[1, 1].set_ylabel('Mean Residual', fontsize=11)
    axes[1, 1].set_title('Average Residuals Across Units', fontsize=12, fontweight='bold')
    axes[1, 1].axhline(y=0, color='k', linestyle='--', linewidth=1)
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_dir / "q2c_model_diagnostics.png", dpi=300, bbox_inches='tight')
    print(f"  Saved: {output_dir / 'q2c_model_diagnostics.png'}")
    plt.close()

    # Figure 3: Comparison with different rho patterns (like Figure 1 in paper)
    patterns = {
        'Constant': np.full(T, 0.9),
        'Sine': 0.5 + 0.4 * np.cos(2 * np.pi * t_grid / 200),
        'Step': 0.9 - 0.5 * (t_grid > T/2).astype(float),
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for idx, (name, pattern) in enumerate(patterns.items()):
        # Simulate and estimate
        y_sim, X_sim = simulate_spatial_data(T, n, W, pattern, beta_true, 
                                            sigma2_true, nu_true)
        
        model_temp = SpatialScoreModel(max_rho=0.99)
        model_temp.fit(y_sim, X_sim, W, verbose=False)
        
        # Plot
        axes[idx].plot(pattern, 'k-', linewidth=2, label='True', alpha=0.7)
        axes[idx].plot(model_temp.filtered_rho, 'r--', linewidth=2, 
                    label='Filtered', alpha=0.7)
        axes[idx].set_xlabel('Time', fontsize=11)
        axes[idx].set_ylabel('ρ(t)', fontsize=11)
        axes[idx].set_title(f'{name} Pattern', fontsize=12, fontweight='bold')
        axes[idx].legend(fontsize=9)
        axes[idx].grid(True, alpha=0.3)
        axes[idx].set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(output_dir / "q2c_pattern_comparison.png", dpi=300, bbox_inches='tight')
    print(f"  Saved: {output_dir / 'q2c_pattern_comparison.png'}")
    plt.close()

    print("Analysis complete")

main_q2c()

Question 2c: Time-Varying Spatial Score Model

Simulation Setup:
  Time periods (T): 500
  Cross-sections (n): 9
  True parameters: ω=0.05, A=0.05, B=0.8
  Student's t df: 5.0
Simulating data from spatial model...
  Data shape: y=(500, 9), X=(500, 9, 3)
Estimating spatial score model...
Starting optimization...
RUNNING THE L-BFGS-B CODE

           * * *



 This problem is unconstrained.


Machine precision = 2.220D-16
 N =            8     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.16568D+04    |proj g|=  1.92402D+04

At iterate    1    f=  1.13925D+04    |proj g|=  1.76553D+03

At iterate    2    f=  1.13240D+04    |proj g|=  1.75404D+03



 Bad direction in the line search;
   refresh the lbfgs memory and restart the iteration.


Optimization completed. Success: False
           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
    8      3     56      2     0     0   1.754D+03   1.132D+04
  F =   11324.007418140056     

ABNORMAL_TERMINATION_IN_LNSRCH                              

Final log-likelihood: -11311.10
Estimation Results:
  Convergence: No
  Log-likelihood: -11311.10

Estimated Parameters:
  ω (omega):  0.0341  (true: 0.0500)
  A:          0.0765  (true: 0.0500)
  B:          0.9968  (true: 0.8000)
  β:          [-0.00089889  0.03038135 -0.00993342]
  σ²:         1.0149  (true: 2.0000)
  ν:          4.9548  (true: 5.0000)

  Unconditional mean of ρ


 Line search cannot locate an adequate point after MAXLS
  function and gradient evaluations.
  Previous x, f and g restored.
 Possible causes: 1 error in function or gradient evaluation;
                  2 rounding error dominate computation.


  Saved: figures_q2/q2c_model_diagnostics.png
  Saved: figures_q2/q2c_pattern_comparison.png
Analysis complete
